# Tutorial: Managing Authorized Views for CCAI Insights

This notebook provides a tutorial on how to use the `authorized_views.py` module. This module simplifies the process of creating and managing Authorized Views within Google Cloud Contact Center AI (CCAI) Insights.

## What are Authorized Views?Authorized Views in CCAI Insights allow you to grant granular access to conversation data. For instance, you can configure views so that individual agents can *only* see the conversations they participated in, rather than all conversations in the project. This is crucial for data privacy, compliance, and focusing agents on their relevant data.

## Setup and DependenciesThe `authorized_views.py` module relies on several libraries, including `pandas` for data manipulation and an internal `conidk` wrapper for Google Cloud services (IAM, CCAI Insights, Google Sheets).**Important:** The `conidk` library is an internal Google wrapper. To make this notebook runnable for demonstration purposes, we will *mock* the `conidk` classes. In a real-world scenario, you would have `conidk` properly configured and authenticated within your Google Cloud environment.First, let's install `pandas`.

In [ ]:
# Install necessary libraries!pip install pandas google-auth -q

### Mocking `conidk` LibraryWe'll create mock implementations of the `conidk.wrapper.iam`, `conidk.wrapper.insights`, and `conidk.wrapper.sheets` classes to allow the `Views` class to function without direct access to the actual internal library.

In [ ]:
import pandas as pdimport osfrom io import StringIOimport sysfrom types import ModuleTypeprint("Setting up mock conidk library...")class MockInsightsAuthorizedViews:    def __init__(self, parent: str):        self.parent = parent        print(f"  MockInsightsAuthorizedViews initialized for parent: {parent}")        self._view_sets = {}        self._views = {}    def get_view_set(self, view_set_id: str):        print(f"  MockInsights: Checking for view set '{view_set_id}'...")        return self._view_sets.get(view_set_id)    def create_view_set(self, authorized_view_set_name: str):        print(f"  MockInsights: Creating view set '{authorized_view_set_name}'...")        if authorized_view_set_name not in self._view_sets:            self._view_sets[authorized_view_set_name] = {                "name": f"{self.parent}/authorizedViewSets/{authorized_view_set_name}",                "display_name": authorized_view_set_name            }            print(f"  MockInsights: View set '{authorized_view_set_name}' created.")            return self._view_sets[authorized_view_set_name]        print(f"  MockInsights: View set '{authorized_view_set_name}' already exists.")        return self._view_sets[authorized_view_set_name]    def create_view(self, authorized_view_set_id: str, display_name: str, conversation_filter: str):        print(f"  MockInsights: Creating view '{display_name}' in view set '{authorized_view_set_id}' with filter '{conversation_filter}'...")        if authorized_view_set_id not in self._view_sets:            print(f"  MockInsights Error: View set '{authorized_view_set_id}' does not exist.")            return None        view_name = f"{self._view_sets[authorized_view_set_id]['name']}/authorizedViews/{display_name.replace(' ', '-')}"        view_data = {            "name": view_name,            "display_name": display_name,            "conversation_filter": conversation_filter        }        self._views[view_name] = view_data        print(f"  MockInsights: View '{display_name}' created.")        return view_dataclass MockIAMPolicy:    def __init__(self, project_id: str):        self.project_id = project_id        self._custom_roles = {}        self._policy_bindings = []        print(f"  MockIAMPolicy initialized for project: {project_id}")    def list_custom_roles(self):        print("  MockIAM: Listing custom roles...")        return list(self._custom_roles.keys())    def create_custom_role(self, role_id: str, title: str, description: str, permissions: list):        print(f"  MockIAM: Creating custom role '{role_id}'...")        if role_id not in self._custom_roles:            self._custom_roles[role_id] = {                "name": f"projects/{self.project_id}/roles/{role_id}",                "title": title,                "description": description,                "permissions": permissions            }            print(f"  MockIAM: Custom role '{role_id}' created.")            return role_id         print(f"  MockIAM: Custom role '{role_id}' already exists.")        return role_id    def get(self):        print("  MockIAM: Getting IAM policy...")        class MockBinding:            def __init__(self, role, members):                self.role = role                self.members = members        return type('Policy', (), {'bindings': [MockBinding(b['role'], b['members']) for b in self._policy_bindings]})()    def add(self, member: str, role: str):        print(f"  MockIAM: Adding member '{member}' to role '{role}'...")        found_binding = False        for binding in self._policy_bindings:            if binding['role'] == role:                if member not in binding['members']:                    binding['members'].append(member)                    print(f"  MockIAM: Added '{member}' to existing role '{role}'.")                else:                    print(f"  MockIAM: Member '{member}' already in role '{role}'.")                found_binding = True                break        if not found_binding:            self._policy_bindings.append({"role": role, "members": [member]})            print(f"  MockIAM: Created new binding for role '{role}' with member '{member}'.")class MockSheets:    def __init__(self, sheet_id: str):        self.sheet_id = sheet_id        print(f"  MockSheets initialized for sheet ID: {sheet_id}")    def to_dataframe(self, sheet_name: str = "0"):        print(f"  MockSheets: Reading sheet '{sheet_name}' from sheet ID '{self.sheet_id}'...")        # Simulate some data for a Google Sheet        data = {            "agent_id": ["agentX_sheets", "agentY_sheets"],            "agent_name": ["Charlie", "Diana"]        }        return pd.DataFrame(data)# Create a mock conidk module structureconidk_mock = ModuleType("conidk")conidk_mock.wrapper = ModuleType("conidk.wrapper")conidk_mock.wrapper.iam = MockIAMPolicyconidk_mock.wrapper.insights = MockInsightsAuthorizedViewsconidk_mock.wrapper.sheets = MockSheetssys.modules["conidk"] = conidk_mocksys.modules["conidk.wrapper"] = conidk_mock.wrapperprint("Mock conidk library setup complete.")

### Provide the `authorized_views.py` Source CodeFor this tutorial to be self-contained and runnable, we'll directly include the source code of `authorized_views.py` here. In a real project, this would typically be a file in your environment that you import.

In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Workflow for Authorized Views in Contact Center AI Insights."""

from typing import Dict, Optional
import pandas as pd
from conidk.wrapper import iam
from conidk.wrapper import insights
from conidk.wrapper import sheets

class Views:
    """A class for all things related to Authorized Views."""

    def __init__(
        self,
        project_id: str,
        location: str,
        view_set_id: Optional[str] = None,
        # custom_role: Optional[str] = None,
    ):

        """Initializes the Views class.
        Args:
            project_id: The Google Cloud project ID.
            location: The Google Cloud location.
        """
        self.project_id = project_id
        self.location = location
        self.parent = f"projects/{project_id}/locations/{location}"
        self.insights = insights.AuthorizedViews(parent=self.parent)
        self.iam = iam.Policy(project_id=project_id)
        self.view_set_id = view_set_id

    def _create_view(
        self,
        authorized_view_set_id: str,
        agent_id: str,
        agent_name: str,
    ) -> Optional[Dict]:

        return self.insights.create_view(
            authorized_view_set_id=authorized_view_set_id,
            display_name=f"agent-{agent_name.lower()}-{agent_id}",
            conversation_filter=f"agent_id = \"{agent_id}\"",
        )

    def _create_default_view_set(
        self,
        authorized_view_set_id: str = "default-viewer"
    ) -> Optional[Dict]:
        """Creates a new Authorized Views Set.
        Args:
            authorized_view_set_id: The ID of the authorized view set.
        Returns:
            The created authorized view set.
        """
        view_set = self.insights.get_view_set(view_set_id=authorized_view_set_id)

        if view_set is None:
            view_set = self.insights.create_view_set(
                authorized_view_set_name=authorized_view_set_id
            )

        return view_set

    def _create_deafult_role(
        self,
        role_id: str = "default-insights-viewer"
    ) -> str:

        roles = self.iam.list_custom_roles()
        role = role_id

        if roles is None or role_id not in roles:
            role = self.iam.create_custom_role(
                role_id = role_id,
                title = "Default Agent Authorized View",
                description= "The minimum required permissions to agents visualize their own data",
                permissions = []
            )

        return role

    def _add_iam_policy(
        self,
        role: str,
        agent_ldap: str
    )-> None:
        member = (
            "principalSet://contactcenterinsights.googleapis.com/projects/"
            f"{self.project_id}/type/AuthorizedView/ancestor.name/"
            f"authorizedViewSets/{self.view_set_id}"
        )
        created = False
        policy = self.iam.get()
        for binding in policy.bindings:
            if binding.role == role and member in binding.members:
                created = True
                break

        if not created:
            self.iam.add(member=member, role=role)

        self.iam.add(member=agent_ldap, role=role)

    def bulk_create_agent_views(
        self,
        authorized_view_set_id: Optional[str],
        source_type: str,
        source_path: str,
        sheet_name: str = "0",
    ):

        """Reads a CSV file or a Google Sheet and creates an authorized view for each agent.
        Args:
            source_type: The type of the source, either 'csv' or 'sheets'.
            source_path: The file path for a CSV or the sheet ID for Google Sheets.
            sheet_name: The name of the sheet to read from (for Google Sheets only).
            authorized_view_set_id: The ID of the authorized view set.
        """

        view_set_id = authorized_view_set_id or "default-viewer"
        self._create_default_view_set(authorized_view_set_id=view_set_id)


        if source_type == "csv":
            df = pd.read_csv(source_path)
        elif source_type == "sheets":
            gsheet = sheets.Sheets(sheet_id=source_path)
            df = gsheet.to_dataframe(sheet_name=sheet_name)
        else:
            raise ValueError("source_type must be either 'csv' or 'sheets'")

        for _, row in df.iterrows():
            agent_id = row["agent_id"]
            agent_name = row["agent_name"]

            self._create_view(
                authorized_view_set_id=view_set_id,
                agent_id=agent_id,
                agent_name=agent_name,
            )


### Google Cloud ConfigurationReplace the placeholders with your Google Cloud Project ID and desired location. If running in Colab, you might need to authenticate your Google Cloud account.

In [ ]:
from google.colab import auth# Authenticate Colab to Google Cloud. This will prompt for credentials.try:    auth.authenticate_user()    print("Google Cloud authenticated.")except Exception as e:    print(f"Authentication failed: {e}. Please ensure you're logged into your Google account.")# Define your Google Cloud project and locationPROJECT_ID = "YOUR_PROJECT_ID" # e.g., "my-gcp-project-12345"LOCATION = "us-central1"      # e.g., "us-central1" or "global"if PROJECT_ID == "YOUR_PROJECT_ID":    print("WARNING: Please update `PROJECT_ID` with your actual Google Cloud Project ID.")print(f"Using Project ID: {PROJECT_ID}")print(f"Using Location: {LOCATION}")

## Initializing the `Views` ClientNow, let's create an instance of the `Views` class. This object will be used to interact with CCAI Insights and IAM to manage authorized views.

In [ ]:
# Initialize the Views clientviews_client = Views(project_id=PROJECT_ID, location=LOCATION)print("\nViews client initialized successfully.")

## Preparing Agent Data for Bulk View CreationThe `bulk_create_agent_views` method requires a list of agents, each with an `agent_id` and `agent_name`. This data can come from a CSV file or a Google Sheet.Let's create a sample CSV file for demonstration.

In [ ]:
# Create a sample CSV file for agent datacsv_data = """agent_id,agent_nameagent123,Aliceagent456,Bobagent789,Eve"""csv_file_path = "agents.csv"with open(csv_file_path, "w") as f:    f.write(csv_data)print(f"Sample agent data saved to '{csv_file_path}':")!cat {csv_file_path}

## Creating Authorized Views in Bulk (`bulk_create_agent_views`)

The `bulk_create_agent_views` method automates the creation of Authorized Views. It performs the following steps:
1.  **Ensures a View Set Exists:** It calls `_create_default_view_set` to ensure the specified `authorized_view_set_id` (defaulting to "default-viewer") exists. This view set acts as a container for related authorized views.
2.  **Reads Agent Data:** It reads the agent information from either a CSV file or a Google Sheet.
3.  **Creates Individual Views:** For each agent in the data, it calls `_create_view`, which creates an authorized view with a `conversation_filter` specific to that agent's ID (e.g., `agent_id = "agent123"`).

In [ ]:
# Define the authorized view set ID (optional, defaults to "default-viewer")AUTH_VIEW_SET_ID = "my-agent-views"print(f"Creating authorized views for agents in view set: {AUTH_VIEW_SET_ID}")views_client.bulk_create_agent_views(    authorized_view_set_id=AUTH_VIEW_SET_ID,    source_type="csv",    source_path=csv_file_path)print("\nBulk creation of agent views complete (mocked).")

### Using Google Sheets as a Source (Optional)If your agent data is in a Google Sheet, you can use `source_type='sheets'`. You'll need the Google Sheet ID and optionally the sheet name. You would also need to ensure that the Colab environment (or your local environment) has the necessary permissions to read from Google Sheets, typically via the Google Drive API.

In [ ]:
# Uncomment and replace with your Google Sheet ID to try this option# YOUR_SHEET_ID = "1ABCDEFG..." # Replace with your Google Sheet ID# views_client.bulk_create_agent_views(#     authorized_view_set_id=AUTH_VIEW_SET_ID,#     source_type="sheets",#     source_path=YOUR_SHEET_ID,#     sheet_name="AgentData" # Optional, defaults to "0"# )# print("Bulk creation of agent views from Google Sheets complete (mocked).")

## Managing IAM Permissions for Authorized ViewsNote that `bulk_create_agent_views` only *creates* the Authorized View definitions. To allow agents to actually *access* these views, you need to manage IAM permissions. This typically involves:
1.  **Creating a Custom IAM Role**: Defining a role with the minimum necessary permissions for agents to view their data in CCAI Insights.
2.  **Adding IAM Policy Bindings**: Associating this role with the Authorized View Set (making the view set a 'principal') and individual agents (e.g., via their LDAP email).

### 1. Creating a Default Custom IAM RoleThe `_create_deafult_role` method can be used to create a custom IAM role with a predefined description and no specific permissions (you would add permissions specific to CCAI Insights viewing if this were a real scenario).

In [ ]:
DEFAULT_ROLE_ID = "default-insights-viewer"created_role_id = views_client._create_deafult_role(role_id=DEFAULT_ROLE_ID)print(f"Custom IAM Role created/verified: {created_role_id}")

### 2. Adding IAM Policy BindingsThe `_add_iam_policy` method adds members (the Authorized View Set and individual agents) to the specified custom role. This grants them the permissions defined in that role for the resources managed by the view set.

**Important**: For `_add_iam_policy` to correctly construct the `principalSet` member, the `views_client.view_set_id` must be set to the ID of the view set you're managing. If you didn't set it during initialization, you can set it directly.

In [ ]:
# Ensure view_set_id is set on the client for IAM operationsviews_client.view_set_id = AUTH_VIEW_SET_IDprint(f"Views client's view_set_id set to: {views_client.view_set_id}")# Example agent LDAP email (replace with a real agent's email in a live system)YOUR_AGENT_LDAP = "user:agent-alice@example.com" if YOUR_AGENT_LDAP == "user:agent-alice@example.com":    print("WARNING: Please update `YOUR_AGENT_LDAP` with a real agent's email for actual IAM binding.")# Add IAM policy binding for the view set (as a principalSet)# This grants the custom role permissions to the entire view setviews_client._add_iam_policy(    role=DEFAULT_ROLE_ID,    agent_ldap=YOUR_AGENT_LDAP # This parameter is actually ignored for the principalSet binding                             # but is used for the subsequent individual agent binding.)# Add IAM policy binding for a specific agent# This grants the custom role permissions directly to an individual userviews_client.iam.add(    member=YOUR_AGENT_LDAP,    role=DEFAULT_ROLE_ID)print("\nIAM policy bindings added (mocked). An agent with the LDAP 'agent-alice@example.com' would now have access to their views through the 'my-agent-views' view set.")

## ConclusionThis tutorial demonstrated how to use the `authorized_views.py` module to:
- Initialize the `Views` client for a specific Google Cloud project and location.
- Prepare agent data from a CSV file.
- Bulk create Authorized Views for multiple agents within a designated View Set.
- Create a custom IAM role and apply policy bindings to grant agents access to their respective views.

By following these steps, you can efficiently manage granular access to Contact Center AI Insights data for your agents, enhancing data security and operational efficiency.